# 1장 3강: 선형 변환과 신경망 레이어 — 실습문제

## 실습 목표

- 같은 위치끼리 곱해 더하여 선형 변환을 계산합니다.
- 편향을 더하는 아핀 변환과 선형 변환의 차이를 확인합니다.
- 가중치와 편향을 각각 바꿀 때 점수가 어떻게 달라지는지 확인합니다.
- ReLU의 규칙을 적용하고, 두 레이어의 계산을 순서대로 따라갑니다.

## 실습 환경 / 데이터

- Python, NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다.
- 제공된 「1장 3강: 선형 변환과 신경망 레이어」의 예제와 연습문제에 있는 숫자·방법을 사용합니다.
- 새로운 공식을 유도하거나 모델을 학습시키는 과제는 없습니다.

## 실습 구성

| 구분 | 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | 선형·아핀 변환과 0벡터 | 1절의 x=[3,2] 예제 |
| 필수 1 / 문제 1-2 | 가중치와 편향을 따로 변경 | 2절·연습문제 2 |
| 필수 2 / 문제 2-1 | ReLU와 한 레이어 계산 | 3절의 고객 3명 예제 |
| 필수 2 / 문제 2-2 | 은닉층에서 출력층으로 전달 | 3절의 전체 순전파 예제 |
| 과제 / 문제 3-1 | 선형·아핀 변환 반복 | 교안 연습문제 1 |
| 과제 / 문제 3-2 | 두 레이어 순전파 반복 | 교안 연습문제 3 |

필수 실습 4문제와 기초 과제 2문제입니다. 과제도 필수 실습에서 사용한 계산 순서를 그대로 적용합니다.

## 진행 안내

1. 위에서 아래로 읽고, 먼저 NumPy를 불러오는 셀을 실행합니다.
2. 각 문제의 숫자와 작성 도움말을 보고 코드를 한 줄씩 작성합니다.
3. 출력값을 확인한 뒤 답변 작성란에 계산 과정과 답을 적습니다.
4. 수식은 먼저 말로 읽고, 곱셈 → 덧셈 → 편향 → ReLU 순서를 확인합니다. ReLU는 적용하라고 한 곳에서만 사용합니다.
5. 변수는 각 문제에서 다시 만듭니다. 앞 문제의 입력값이나 가중치를 그대로 쓰지 않도록 주의하세요.

## 처음 보는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 입력 / 출력 | 계산에 넣는 숫자 / 계산해서 얻는 숫자 |
| 벡터 / 행렬 | 숫자를 한 줄로 나열한 것 / 숫자를 행과 열로 놓은 것 |
| 행 / 열 | 가로로 읽는 줄 / 세로로 읽는 줄 |
| 특성 | 고객을 나타내는 한 가지 정보. 예: 방문 횟수 |
| x / X | 한 고객의 입력 / 여러 고객의 입력을 모은 배열 |
| w / W | 입력값에 곱할 가중치. W의 각 열은 출력 하나에 사용할 가중치 |
| b / bias | 곱하고 더한 결과에 마지막으로 더할 편향 |
| 파라미터 | 이 강의에서는 가중치와 편향을 가리키는 말 |
| 뉴런 / 레이어 | 입력으로 출력 하나를 만드는 계산 단위 / 뉴런들을 모은 계산 단계 |
| 은닉층 / 출력층 | 입력과 최종 결과 사이의 계산 단계 / 최종 결과를 만드는 단계 |
| Z, Z1 | 가중치를 곱해 더하고 편향까지 더한 값 |
| A, H | ReLU를 적용한 값. 고객 이름 A와 코드 변수 A는 문맥으로 구분 |
| Y | 최종 출력값을 모은 배열 |
| Shape | 배열의 모양. (3,2)는 3행 2열, (2,)는 숫자 2개인 1차원 배열 |

`xW`는 수식에서 x와 W의 행렬곱을 뜻하며, 코드에서는 `x @ W`로 씁니다. `xW+b`는 그 결과에 편향 b를 더하라는 뜻입니다. `T(x)`는 ‘입력 x에 T라는 계산을 적용한 결과’로 읽습니다.

##### 활성화 함수
1. 가정: 공부를 시작한 뒤 처음 2시간은 기초를 익히기 때문에 점수 상승을 기대하기 어렵다. 기초를 충분히 파악한 2시간 뒤부터 점수가 올라갈 것이다.
- 계산식: h = x -2 | y = 50 + 10h
- y = 50 + 10(x-2)
- ( y = 30 + 10x ) <<- 이걸 방지하기 위해서

2. ReLU 도입:
- h = ReLu(x-2) = max(0, x-2)
- 공부 시간이 0시간일 때, x-2 = -2이지만 ReLU를 적용하면 0이므로 최종 점수 50점을 깎아먹지는 않는다.
- 공부 시간이 1시간일 때, x-2 = -1이지만 ReLU를 적용하면 0이므로 최종 점수 50점을 깎아먹지는 않는다.
- 공부 시간이 2.000000000000000001 시간일 때, x-2 = 0.0000000000000000001이므로 ReLU를 적용하면 최종 점수는 50.000000000000000001이 된다.

6. 결론: 
- 2시간 이하의 공부 시간이라면 굳이 패턴을 학습하지 않고 50점 유지
- 2시간 초과의 공부 시간이라면 시간에 따른 점수 증가
- 모델의 복잡도를 낮추고 다음 레이어의 계산식을 해치지 않기 위한 branch같은 개념

##### 공부 기록으로 시험 점수 예측하는 신경망 만들기
- 입력값 3개 (공부 시간, 풀어본 문제 수, 복습 횟수)
- (2, 3, 1)로 가정
- 신경망에 사용되는 데이터 X1, X2, X3 = [2,3,1]
####
- 은닉층은 2개, 뉴런은 3개를 사용하여 딥러닝을 구성해보겠다.
- 첫번째 은닉층, 뉴런 3개가 각각 계산하겠다.
- 세 뉴런 모두가 X의 특성값인 [2,3,1]을 받는다. 단, 각각의 뉴런은 가중치와 편향이 다르기 때문에 다른 결과를 얻는다.
####
- 첫 번째 뉴런, 가중치는 [1, 1, -1] 편향은 -1로 가정하면
- z1 = (1x2) + (1x3) + (-1x1) -1 = 3
- h1 = ReLU(3) = 3
####
- 두 번째 뉴런, 가중치는 [-1, 1, 1] 편향은 0으로 가정
- z2 = (-1x2) + (1x3) + (1x1) + 0 = 2
- h2 = ReLU(2) = 2
####
- 세 번째 뉴런, 가중치는 [1, -1, -1] 편향은 1로 가정
- z3 = (1x2) + (-1x3) + (-1x1) + 1 = -1
- h3 = ReLU(-1) = 0

|뉴런|가중합 + 편향|ReLU 적용 후|
|---|---|---|
|뉴런1|3|3|
|뉴런2|2|2|
|뉴런3|-1|0|

#### 첫 번째 은닉층의 출력값은 [3,2,0]이다.
- 두 번째 은닉층에서 사용할 중간값을 의미한다.
####
#### 두 번째 은닉층: 앞에서 나온 중간값 [3,2,0]을 받는다.
- 첫 번째 뉴런: 첫 번째 은닉층은 원래 데이터 [2,3,1]을 받았지만 두 번째 은닉층에서는 첫 번째 은닉층의 결과인 [3,2,0]을 받아서 계산한다.
- u1 = (1x3) + (1x2) + (1x0) -1 = 4
- a1 = ReLU(4) = 4
#### 
- 두 번째 뉴런: 
- u2 = (-1x3) + (1x2) + (1x0) + 0 = -1
- a2 = ReLU(-1) = 0
####
- 세 번째 뉴런:
- u3 = (1x3) + (-1x2) + (1x0) + 1 = 2
- a3 = ReLU(2) = 2
#### 두 번째 은닉층의 출력값은 [4,0,2]이다.

|뉴런|가중합 + 편향|ReLU 적용 후|
|---|---|---|
|뉴런1|4|4|
|뉴런2|-1|0|
|뉴런3|2|2|

####
#### 두 개의 은닉층을 활용하여 최종 출력층을 만들어본다면
- 출력층은 마지막(두 번째 은닉층)에서 나온 [4,0,2]를 받는다.
- 출력층도 마찬가지로 뉴런의 가중치를 [10, -5, 5], 편향을 20이라고 가정해보면
####
- y = (10x4) + (-5x0) + (5x2) + 20 = 70점
- 최종 예측 점수는 70점으로 볼 수 있다. 

#### 지금까지의 순서 정리
1. 입력층: 학생의 정보 전달 [2,3,1]
2. 첫 번째 은닉층: 뉴런 3개가 각 가중합과 편향 계산 [3,2,-1]
3. 첫 번째 은닉층의 ReLU: 음수 결과를 0으로 변경하여 [3,2,0]
4. 두 번째 은닉층: 뉴런 3개가 각 가중합과 편향을 계산 [4,-1,2]
5. 두 번째 은닉층의 ReLU: 음수 결과를 0으로 변경하여 [4,0,2]
6. 출력층: 마지막 가중합과 편향을 계산 -> 예측 점수 70점
####
- 출력층의 예측 점수와 실제 점수를 비교한 뒤 가중치와 편향을 수정한다.
- 이 학생의 실제 점수가 80점이었다면 예측값 70점, 실제 점수 80점이므로
- 제곱 오차 = (70 - 80)² = 100이다.
- 이 오차를 바탕으로 다시 각 가중치와 편향의 기울기를 조정?
- 이 오차를 다시 출력층이나 은닉층으로 보내서 계산시키는 것 = 역전파

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 배열을 계산하는 도구이며, `np`는 코드에서 NumPy를 짧게 부르는 이름입니다.

In [1]:
import numpy as np

---

## 필수 1. 선형·아핀 변환과 파라미터 이해하기

### 문제 1-1. 선형 변환에 편향을 더해 보기

#### 문제 설명

교안 1절의 예제입니다. 선형 변환은 입력에 가중치 행렬을 곱하는 계산이며, 아핀 변환은 그 결과에 편향을 더하는 계산입니다.

```python
x = np.array([3, 2])
W = np.array([[2, 0], [0, 3]])
bias = np.array([1, -2])
zero = np.array([0, 0])
```

- 선형 변환: `x @ W`
- 아핀 변환: `x @ W + bias`

W의 첫 번째 열은 [2,0], 두 번째 열은 [0,3]입니다. 0벡터는 모든 값이 0인 벡터를 말합니다.

#### 요구사항

1. x, W, bias, zero를 위와 같이 만드세요.
2. `linear_output = x @ W`, `affine_output = x @ W + bias`를 계산하여 출력하세요.
3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
4. 답변 작성란에 두 출력값의 곱셈·덧셈 과정을 적으세요.

#### 작성 도움말

첫 출력은 x의 [3,2]와 W의 첫 열 [2,0]을 같은 순서로 짝지어 `3×2 + 2×0`으로 계산합니다. 둘째 출력도 둘째 열을 사용해 같은 방법으로 계산하세요. 편향은 첫 출력에 1, 둘째 출력에 −2를 각각 더합니다.

#### 해석 질문

**Q1.** 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?

**Q2.** 편향을 더한 아핀 변환 결과는 무엇인가요?

**Q3.** 0벡터 입력에서 두 변환의 결과는 각각 무엇인가요?

**Q4.** 편향이 [0,0]이라면 두 변환의 결과가 같아지나요?

#### 제출 결과

- 두 변환의 출력과 0벡터의 출력
- 계산 과정과 Q1~Q4 답변

In [4]:
# 문제 1-1. 선형 변환에 편향을 더해 보기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.
# 1. x, W, bias, zero를 위와 같이 만드세요.
# 2. `linear_output = x @ W`, `affine_output = x @ W + bias`를 계산하여 출력하세요.
# 3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
# 4. 답변 작성란에 두 출력값의 곱셈·덧셈 과정을 적으세요.

x = np.array([3, 2])
W = np.array([[2, 0], [0, 3]])
bias = np.array([1, -2])
zero = np.array([0, 0])

linear_output = x @ W
affine_output = x @ W + bias

print("요구사항 2")
print(linear_output)
print(affine_output)
print("----------------------------------")
print("요구사항 3")
print(zero @ W)
print(zero @ W + bias)
print("----------------------------------")

요구사항 2
[6 6]
[7 4]
----------------------------------
요구사항 3
[0 0]
[ 1 -2]
----------------------------------


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?**

- 답변: 3x2 + 2x0 = 6 | 3x0 + 2x3 = 6이므로 [6, 6]

**Q2. 편향을 더한 아핀 변환 결과는 무엇인가요?**

- 답변: 3x2 + 2x0 + 1 = 7 | 3x0 + 2x3 + (-2) = 4 이므로 [7, 4] 

**Q3. 0벡터 입력에서 두 변환의 결과는 각각 무엇인가요?**

- 답변: 
- 선형:2x0 + 0x0 = 0 | 0x0 + 3x0 = 0 이므로 [0, 0]
- 아핀:2x0 + 0x0 + 1 = 1 | 0x0 + 3x0 -2 = -2 이므로 [1, -2]

**Q4. 편향이 [0,0]이라면 두 변환의 결과가 같아지나요?**

- 답변: 네. 0벡터에 편향도 [0, 0]이라면 두 변환의 결과가 같아집니다.

### 가중치 행렬의 열마다 곱해 더하면 선형변환, 그 결과에 편향을 더하면 더하면 아핀 변환

### 문제 1-2. 가중치와 편향을 하나씩 바꾸기

#### 문제 설명

교안 연습문제 2의 데이터입니다. 각 고객의 첫 입력은 방문 횟수, 둘째 입력은 장바구니 추가 횟수입니다.

```python
X = np.array([[4, 2], [1, 5]])
w = np.array([2, -1])
b = 3
```

X의 첫 행 [4,2]는 고객 A, 둘째 행 [1,5]는 고객 B입니다. 고객별 점수는 ‘첫 입력×첫 가중치 + 둘째 입력×둘째 가중치 + 편향’으로 계산합니다.

| 비교 | 가중치 | 편향 |
|---|---|---|
| 기본 | [2,−1] | 3 |
| 가중치만 변경 | [2,−2] | 3 |
| 편향만 변경 | [2,−1] | 5 |

두 변경은 각각 기본 설정에서 출발합니다.

#### 요구사항

1. 위 배열을 만들고 `base_scores = X @ w + b`를 계산하여 출력하세요.
2. `w_new = np.array([2, -2])`로 만들고 `weight_scores = X @ w_new + b`를 계산하세요. 점수와 `weight_scores - base_scores`를 출력하세요.
3. 기본 w를 사용하여 `b_new = 5`, `bias_scores = X @ w + b_new`를 계산하세요. 점수와 `bias_scores - base_scores`를 출력하세요.
4. `zero = np.array([0, 0])`를 만들고 기본 설정인 `zero @ w + b`를 출력하세요.

#### 작성 도움말

변화량은 ‘변경 후 점수 − 기본 점수’입니다. 변화량이 −2라면 점수가 2만큼 줄었다는 뜻입니다. 편향만 바꿀 때는 w_new가 아닌 기본 w를 사용하세요. 한 번에 한 조건만 바꿉니다.

#### 해석 질문

**Q1.** 기본 설정에서 고객 A와 B의 점수는 각각 얼마인가요?

**Q2.** 가중치만 바꾸면 각 점수와 변화량은 얼마인가요?

**Q3.** 편향만 바꾸면 각 점수와 변화량은 얼마인가요?

**Q4.** 기본 설정에 0벡터를 넣으면 점수가 얼마인가요? 왜 그 값이 남나요?

#### 제출 결과

- 기본 점수, 두 변경의 점수와 변화량, 0벡터의 점수
- 계산 과정과 Q1~Q4 답변

In [9]:
# 문제 1-2. 가중치와 편향을 하나씩 바꾸기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.
# 1. 위 배열을 만들고 `base_scores = X @ w + b`를 계산하여 출력하세요.
# 2. `w_new = np.array([2, -2])`로 만들고 `weight_scores = X @ w_new + b`를 계산하세요. 점수와 `weight_scores - base_scores`를 출력하세요.
# 3. 기본 w를 사용하여 `b_new = 5`, `bias_scores = X @ w + b_new`를 계산하세요. 점수와 `bias_scores - base_scores`를 출력하세요.
# 4. `zero = np.array([0, 0])`를 만들고 기본 설정인 `zero @ w + b`를 출력하세요.

X = np.array([[4, 2], [1, 5]])
w = np.array([2, -1])
b = 3

print("요구사항 1")
base_scores = X @ w + b
print(base_scores)
print("---------------------------------")

w_new = np.array([2, -2])
weight_scores = X @ w_new + b
print("요구사항 2")
print(weight_scores)
print(weight_scores - base_scores)
print("---------------------------------")

print("요구사항 3")
b_new = 5
bias_scores = X @ w + b_new
print(bias_scores)
print(bias_scores - base_scores)
print("---------------------------------")

zero = np.array([0, 0])
print("요구사항 4")
print(zero @ w + b)


요구사항 1
[9 0]
---------------------------------
요구사항 2
[ 7 -5]
[-2 -5]
---------------------------------
요구사항 3
[11  2]
[2 2]
---------------------------------
요구사항 4
3


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 기본 설정에서 고객 A와 B의 점수는 각각 얼마인가요?**

- 답변: 9점, 0점

**Q2. 가중치만 바꾸면 각 점수와 변화량은 얼마인가요?**

- 답변: 7점, -5점이고 변화량은 -2점, -5점이다.

**Q3. 편향만 바꾸면 각 점수와 변화량은 얼마인가요?**

- 답변: 11점, 2점이고 변화량은 2점, 2점이다

**Q4. 기본 설정에 0벡터를 넣으면 점수가 얼마인가요? 왜 그 값이 남나요?**

- 답변: 3점. 0벡터이므로 편향만 남는다.

---

## 필수 2. ReLU와 신경망 순전파

### 문제 2-1. ReLU와 한 레이어를 순서대로 계산하기

#### 문제 설명

교안 3절의 예제입니다. 먼저 ReLU의 규칙을 연습한 뒤 레이어 하나를 계산합니다.

**ReLU(렐루)의 규칙:** 양수는 그대로 두고, 음수는 0으로 바꿉니다. 0은 0으로 둡니다.

```python
def relu(z):
    return np.maximum(0, z)

test_z = np.array([-3, -1, 0, 2, 5])
X = np.array([[3, 2], [1, 4], [0, 2]])
W = np.array([[1, -2], [2, 1]])
b = np.array([-4, 1])
```

X의 세 행은 고객 A, B, C입니다. W의 두 열은 뉴런 두 개의 가중치입니다. 첫 뉴런의 가중치는 [1,2], 둘째 뉴런의 가중치는 [−2,1]입니다. 편향은 각각 −4와 1입니다.

이번 계산은 **가중치 적용 → 편향 덧셈 → ReLU**의 세 단계입니다.

#### 요구사항

1. 위 코드를 입력하고 `relu(test_z)`를 출력하세요.
2. `weighted = X @ W`를 계산하여 출력하세요.
3. `Z = weighted + b`를 계산하여 출력하세요.
4. `A = relu(Z)`를 계산하여 출력하세요.
5. `Z.shape`와 `A.shape`를 출력하고, 고객 A의 두 뉴런 계산을 손으로 확인하세요.

#### 작성 도움말

`def relu(z):`와 그 아래 `return` 줄은 위 예제를 그대로 사용하세요. return 앞의 들여쓰기도 유지하세요. 고객 A의 첫 뉴런은 `3×1 + 2×2`를 구한 뒤 −4를 더합니다. 마지막에 그 값과 0 중 큰 값을 고릅니다.

#### 해석 질문

**Q1.** ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?

**Q2.** 고객 A의 가중합, 편향을 더한 값 Z, ReLU 결과 A는 각각 무엇인가요?

**Q3.** 고객 B와 C의 Z와 ReLU 결과는 각각 무엇인가요?

**Q4.** Z와 A의 Shape (3,2)는 무엇을 뜻하나요? 음수를 0으로 바꾸면 배열의 크기도 줄어드나요?

#### 제출 결과

- ReLU 연습 결과, weighted·Z·A의 출력, Z·A의 Shape
- 고객 A의 계산 과정과 Q1~Q4 답변

In [13]:
# 문제 2-1. ReLU와 한 레이어를 순서대로 계산하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.
# 1. 위 코드를 입력하고 `relu(test_z)`를 출력하세요.
# 2. `weighted = X @ W`를 계산하여 출력하세요.
# 3. `Z = weighted + b`를 계산하여 출력하세요.
# 4. `A = relu(Z)`를 계산하여 출력하세요.
# 5. `Z.shape`와 `A.shape`를 출력하고, 고객 A의 두 뉴런 계산을 손으로 확인하세요.

def relu(z):
    return np.maximum(0, z)

test_z = np.array([-3, -1, 0, 2, 5])
X = np.array([[3, 2], [1, 4], [0, 2]])
W = np.array([[1, -2], [2, 1]])
b = np.array([-4, 1])

print("요구사항 1")
print(relu(test_z))
print("------------------------------")
print("요구사항 2")
weighted = X @ W
print(weighted)
print("------------------------------")
print("요구사항 3")
Z = weighted + b
print(Z)
print("------------------------------")
print("요구사항 4")
A = relu(Z)
print(A)
print("------------------------------")
print("요구사항 5")
print(Z.shape)
print(A.shape)



요구사항 1
[0 0 0 2 5]
------------------------------
요구사항 2
[[ 7 -4]
 [ 9  2]
 [ 4  2]]
------------------------------
요구사항 3
[[ 3 -3]
 [ 5  3]
 [ 0  3]]
------------------------------
요구사항 4
[[3 0]
 [5 3]
 [0 3]]
------------------------------
요구사항 5
(3, 2)
(3, 2)


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?**

- 답변: 0이하에서 0, 0 이상에서는 그대로 출력하여 [0 0 0 2 5]로 변환한다.

**Q2. 고객 A의 가중합, 편향을 더한 값 Z, ReLU 결과 A는 각각 무엇인가요?**

- 답변: Z = [ 3 -3]  | ReLU 결과 A = [ 3  0]으로, 음수 부분이 0으로 변환되었다.

**Q3. 고객 B와 C의 Z와 ReLU 결과는 각각 무엇인가요?**

- 답변: ReLU B = [5, 3] | ReLU C = [0, 3]

**Q4. Z와 A의 Shape (3,2)는 무엇을 뜻하나요? 음수를 0으로 바꾸면 배열의 크기도 줄어드나요?**

- 답변: shape (3,2)는 결과 행렬이 3행 2열짜리 행렬임을 뜻한다. 3은 고객 3명, 2는 고객마다 뉴런 2개가 만든 출력의 수 이다.
- ReLU를 통해 음수를 0으로 바꿔도 배열의 크기는 줄어들지 않는다. 

### 문제 2-2. 은닉층의 결과를 출력층에 전달하기

#### 문제 설명

교안 3절의 전체 순전파 예제입니다. 순전파란 입력부터 출력까지 앞 단계의 결과를 다음 단계로 전달하며 계산하는 과정입니다.

```python
X = np.array([[3, 2], [1, 4], [0, 2]])
W1 = np.array([[1, -2], [2, 1]])
b1 = np.array([-4, 1])
W2 = np.array([[2], [-1]])
b2 = np.array([1])
```

첫 단계인 은닉층은 문제 2-1과 같습니다. 은닉층의 뉴런은 2개이고, 그 뒤에 뉴런 1개인 출력층을 붙입니다. 아래 세 줄을 위에서부터 계산하세요.

```python
Z1 = X @ W1 + b1
H = relu(Z1)
Y = H @ W2 + b2
```

Z1은 은닉층에서 편향까지 더한 값, H는 ReLU를 적용한 값, Y는 최종 점수입니다. 은닉층에는 ReLU를 적용하고, 이 교안 예제의 출력층에는 별도 활성화 함수를 적용하지 않습니다.

#### 요구사항

1. 문제 2-1의 relu 함수를 그대로 작성하고, 위 입력과 파라미터를 만드세요.
2. Z1, H, Y를 제시된 순서대로 계산하여 출력하세요.
3. `H.shape`와 `Y.shape`를 출력하세요.
4. 고객별 H의 두 숫자를 사용해 최종 점수를 손으로 계산하세요.

#### 작성 도움말

W2의 첫 값은 2, 둘째 값은 −1입니다. 따라서 H의 한 행이 [h₁,h₂]이면 최종 점수는 `h₁×2 + h₂×(−1) + 1`입니다. h₁과 h₂는 각각 그 행의 첫째·둘째 숫자를 가리킵니다. 출력층에는 X나 Z1 대신 H를 넣으세요.

#### 해석 질문

**Q1.** Z1과 H는 어떻게 다른가요? 출력층의 입력으로 어느 것을 사용하나요?

**Q2.** 고객 A·B·C의 최종 점수는 각각 얼마인가요? 계산 과정을 적으세요.

**Q3.** H의 Shape (3,2)와 Y의 Shape (3,1)는 각각 무엇을 뜻하나요?

**Q4.** 고객 C의 최종 점수가 음수로 나와도 되는 이유는 무엇인가요?

#### 제출 결과

- Z1·H·Y 출력과 H·Y의 Shape
- 고객별 최종 계산 과정과 Q1~Q4 답변

In [19]:
# 문제 2-2. 은닉층의 결과를 출력층에 전달하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.
# 1. 문제 2-1의 relu 함수를 그대로 작성하고, 위 입력과 파라미터를 만드세요.
# 2. Z1, H, Y를 제시된 순서대로 계산하여 출력하세요.
# 3. `H.shape`와 `Y.shape`를 출력하세요.
# 4. 고객별 H의 두 숫자를 사용해 최종 점수를 손으로 계산하세요.

def relu(z):
    return np.maximum(0, z)

X = np.array([[3, 2], [1, 4], [0, 2]])
W1 = np.array([[1, -2], [2, 1]])
b1 = np.array([-4, 1])
W2 = np.array([[2], [-1]])
b2 = np.array([1])

Z1 = X @ W1 + b1
H = relu(Z1)
Y = H @ W2 + b2

print("요구사항 2")
print("Z1:", Z1)
print("H:", H)
print("Y:", Y)
print("-------------------------------")
print("요구사항 3")
print("H shape:", H.shape)
print("Y shape:", Y.shape)
print("-------------------------------")


요구사항 2
Z1: [[ 3 -3]
 [ 5  3]
 [ 0  3]]
H: [[3 0]
 [5 3]
 [0 3]]
Y: [[ 7]
 [ 8]
 [-2]]
-------------------------------
요구사항 3
H shape: (3, 2)
Y shape: (3, 1)
-------------------------------


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. Z1과 H는 어떻게 다른가요? 출력층의 입력으로 어느 것을 사용하나요?**

- 답변: Z1은 가중합이고 H는 그 가중합에 ReLU를 적용한 것이다. 출력층의 입력으로는 H를 사용한다.

**Q2. 고객 A·B·C의 최종 점수는 각각 얼마인가요? 계산 과정을 적으세요.**

- 답변: A: 3x2 + 0x(-1) + 1 = 7 | B: 5x2 + 3x(-1) + 1 = 8 | C: 0x2 + 3x(-1) + 1 = -2

**Q3. H의 Shape (3,2)와 Y의 Shape (3,1)는 각각 무엇을 뜻하나요?**

- 답변: H shape은 은닉층의 계산 결과인 중간값이 (3,2)배열인 행렬임을 뜻하고(고객 3명의 은닉층 출력이 2개씩 있다.)
- Y shape은 최종 출력층의 결과가 (3,1)배열의 행렬 즉, 고객 3명의 최종 점수임을 뜻한다.

**Q4. 고객 C의 최종 점수가 음수로 나와도 되는 이유는 무엇인가요?**

- 답변: 이번 실습에서는 은닉층에만 ReLU를 적용하고 출력층에는 적용하지 않았다.
- C의 [0,3]은 음수가 없지만 출력층에서 3에 -1을 곱하면 -3이 된다.
- 출력층의 가중치는 음수가 될 수 있기 때문이다.

---

## 과제. 필수 실습과 같은 방법 적용하기

### 문제 3-1. 선형 변환과 아핀 변환 다시 계산하기

#### 문제 설명

교안 연습문제 1입니다. 문제 1-1과 같은 방법을 사용하세요.

```python
x = np.array([2, 3])
W = np.array([[2, 0], [1, 3]])
bias = np.array([1, -2])
zero = np.array([0, 0])
```

W의 첫 열은 [2,1], 둘째 열은 [0,3]입니다. 이번에는 첫 열의 두 번째 가중치가 0이 아닌 1이라는 점을 확인하세요.

#### 요구사항

1. 위 배열을 만들고 `linear_output = x @ W`를 계산하여 출력하세요.
2. `affine_output = x @ W + bias`를 계산하여 출력하세요.
3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
4. 각 출력값에 사용한 곱셈과 덧셈을 답변 작성란에 적으세요.

#### 작성 도움말

첫 출력은 `2×2 + 3×1`입니다. 둘째 출력은 둘째 열 [0,3]을 이용하세요. 선형 변환을 모두 구한 뒤 각 위치의 편향을 더합니다. 새로운 함수나 공식을 찾을 필요 없이 문제 1-1의 순서대로 계산하세요.

#### 해석 질문

**Q1.** 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?

**Q2.** 편향을 더한 아핀 변환 결과는 무엇인가요?

**Q3.** 0벡터 입력에서 선형 변환과 아핀 변환 결과는 각각 무엇인가요?

#### 제출 결과

- 선형·아핀 변환 및 0벡터의 출력
- 계산 과정과 Q1~Q3 답변

In [20]:
# 문제 3-1. 선형 변환과 아핀 변환 다시 계산하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.
# 1. 위 배열을 만들고 `linear_output = x @ W`를 계산하여 출력하세요.
# 2. `affine_output = x @ W + bias`를 계산하여 출력하세요.
# 3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
# 4. 각 출력값에 사용한 곱셈과 덧셈을 답변 작성란에 적으세요.

# 배열 생성
x = np.array([2, 3])
W = np.array([[2, 0], [1, 3]])
bias = np.array([1, -2])
zero = np.array([0, 0])

# 1. linear_output = x @ W 계산 및 출력
linear_output = x @ W
print("1. 선형 변환 출력 (linear_output):", linear_output)
print("-" * 45)

# 2. affine_output = x @ W + bias 계산 및 출력
affine_output = x @ W + bias
print("2. 아핀 변환 출력 (affine_output):", affine_output)
print("-" * 45)

# 3. zero @ W와 zero @ W + bias 출력
print("3. 0벡터 입력 결과:")
print("zero @ W (선형 변환):", zero @ W)
print("zero @ W + bias (아핀 변환):", zero @ W + bias)

1. 선형 변환 출력 (linear_output): [7 9]
---------------------------------------------
2. 아핀 변환 출력 (affine_output): [8 7]
---------------------------------------------
3. 0벡터 입력 결과:
zero @ W (선형 변환): [0 0]
zero @ W + bias (아핀 변환): [ 1 -2]


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?**

- 답변: 2x2 + 3x1 = 7 | 2x0 + 3x3 = 9이므로 [7, 9]

**Q2. 편향을 더한 아핀 변환 결과는 무엇인가요?**

- 답변: 2x2 + 3x1 + 1 = 8 | 2x0 + 3x3 +(-2) = 7 이므로 [8, 7]

**Q3. 0벡터 입력에서 선형 변환과 아핀 변환 결과는 각각 무엇인가요?**

- 답변: 
- 선형: [0, 0]
- 아핀: [1, -2]

### 문제 3-2. 같은 순서로 두 레이어 계산하기

#### 문제 설명

교안 연습문제 3입니다. 문제 2-2와 같은 순서를 적용합니다.

```python
X = np.array([[2, 1], [1, 3]])
W1 = np.array([[1, -1], [2, 1]])
b1 = np.array([-3, 0])
W2 = np.array([[2], [-1]])
b2 = np.array([1])
```

첫 행은 고객 A, 둘째 행은 고객 B입니다. 은닉층에서 ReLU를 적용하고, 출력층에는 별도 활성화 함수를 적용하지 않습니다.

#### 요구사항

1. 문제 2-1의 relu 함수를 작성하고 위 배열을 만드세요.
2. `Z1 = X @ W1 + b1`을 계산하여 출력하세요.
3. `H = relu(Z1)`을 계산하여 출력하세요.
4. `Y = H @ W2 + b2`와 `Y.shape`를 출력하세요.
5. 아래 질문 순서대로 두 고객의 계산 과정을 적으세요.

#### 작성 도움말

은닉층 첫 뉴런은 첫 열 [1,2]를 사용하고 편향 −3을 더합니다. 둘째 뉴런은 둘째 열 [−1,1]을 사용하고 편향 0을 더합니다. 그 다음 ReLU를 적용하세요. 출력층에서는 H의 첫 값×2 + 둘째 값×(−1) + 1을 계산합니다.

#### 해석 질문

**Q1.** 고객 A의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?

**Q2.** 고객 B의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?

**Q3.** 고객 A와 B의 최종 점수는 각각 얼마인가요? H의 값을 사용해 계산하세요.

**Q4.** 최종 출력의 Shape는 무엇이고, 각 숫자는 무엇을 뜻하나요?

#### 제출 결과

- Z1·H·Y 출력과 Y의 Shape
- 고객별 단계별 계산과 Q1~Q4 답변

In [21]:
# 문제 3-2. 같은 순서로 두 레이어 계산하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

# 1. 문제 2-1의 relu 함수를 작성하고 위 배열을 만드세요.
# 2. `Z1 = X @ W1 + b1`을 계산하여 출력하세요.
# 3. `H = relu(Z1)`을 계산하여 출력하세요.
# 4. `Y = H @ W2 + b2`와 `Y.shape`를 출력하세요.
# 5. 아래 질문 순서대로 두 고객의 계산 과정을 적으세요.

# 1. relu 함수 작성 및 배열 생성
def relu(x):
    return np.maximum(0, x)

X = np.array([[2, 1], [1, 3]])
W1 = np.array([[1, -1], [2, 1]])
b1 = np.array([-3, 0])
W2 = np.array([[2], [-1]])
b2 = np.array([1])

# 2. Z1 = X @ W1 + b1 계산 및 출력
Z1 = X @ W1 + b1
print("2. 은닉층 선형+편향 결과 (Z1):\n", Z1)
print("-" * 40)

# 3. H = relu(Z1) 계산 및 출력
H = relu(Z1)
print("3. 은닉층 활성화 결과 (H = relu(Z1)):\n", H)
print("-" * 40)

# 4. Y = H @ W2 + b2와 Y.shape 출력
Y = H @ W2 + b2
print("4. 출력층 최종 결과 (Y):\n", Y)
print("Y의 Shape:", Y.shape)

2. 은닉층 선형+편향 결과 (Z1):
 [[ 1 -1]
 [ 4  2]]
----------------------------------------
3. 은닉층 활성화 결과 (H = relu(Z1)):
 [[1 0]
 [4 2]]
----------------------------------------
4. 출력층 최종 결과 (Y):
 [[3]
 [7]]
Y의 Shape: (2, 1)


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 고객 A의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?**

- 답변: 
- Z1 = [ 1 -1] | H = [1 0]

**Q2. 고객 B의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?**

- 답변: 
- Z1 = [ 4 2] | H = [4 2]

**Q3. 고객 A와 B의 최종 점수는 각각 얼마인가요? H의 값을 사용해 계산하세요.**

- 답변: [[3], [7]] (A 3점, B 7점)

**Q4. 최종 출력의 Shape는 무엇이고, 각 숫자는 무엇을 뜻하나요?**

- 답변: (2,1)이고 고객 2명의 최종 점수를 뜻한다.

---

## 실습 마무리

1. 선형 변환과 아핀 변환의 계산은 어떻게 다른가요?
2. 가중치와 편향은 각각 언제 사용하는 숫자인가요?
3. ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?
4. 두 레이어의 계산에서 출력층에 전달하는 값은 무엇인가요?
5. 최종 출력의 Shape (3,1)는 무엇을 뜻하나요?
6. 주어진 가중치와 편향으로 출력만 계산하면 학습이 이루어지나요?

### 실습 마무리 답변 작성란

**1. 선형 변환과 아핀 변환의 계산은 어떻게 다른가요?**

- 답변: 선형 변환의 계산식 x @ W
- 아핀 변환은 선형 변환의 결과에 편향을 더한 x @ W + b
- 예시로 [6,6]에 편향 [1,-2]를 더하여 최종 [7,4]를 얻음

**2. 가중치와 편향은 각각 언제 사용하는 숫자인가요?**

- 답변: 가중치는 각 입력에 곱하는 숫자, 편향은 곱셈 결과들을 더한 뒤 추가로 더하는 숫자
- 4x2 + 2x(-1) + 3에서 가중치는 2와 -1이고 편향은 3이다.

**3. ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?**

- 답변: [0, 0, 0, 2, 5] 
- 음수를 0으로 바꾼다.

**4. 두 레이어의 계산에서 출력층에 전달하는 값은 무엇인가요?**

- 답변: 중간값= 은닉층의 가중합에 편향을 더한 Z1에 ReLU를 적용하여 만든 H
- 순서는 Z1 = W1 @ b1 | H = ReLU(Z1) | Y = H @ W2 + B2

**5. 최종 출력의 Shape (3,1)는 무엇을 뜻하나요?**

- 답변: 3행 1열, 고객 3명으로부터 여러 피쳐를 통해 하나의 값을 얻었다.

**6. 주어진 가중치와 편향으로 출력만 계산하면 학습이 이루어지나요?**

- 답변: 아니다. 주어진 값으로 출력까지 계산하는 과정은 순전파